# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided as a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
We can load the dataset's metadata and access data records using `mlcroissant`. The Croissant schema provides structured definitions for record sets, fields, and related resources.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Dataset Description: {metadata.description}\n")

# Print high-level information
if hasattr(metadata, 'keywords'):
    print('Keywords:', ', '.join(metadata.keywords))
if hasattr(metadata, 'license'):
    print('License:', metadata.license)
if hasattr(metadata, 'conformsTo'):
    print('Croissant Schema Version:', metadata.conformsTo)
if hasattr(metadata, 'datePublished'):
    print('Published:', metadata.datePublished)

## 2. Data Overview
Let's inspect available record sets and their `@id`s. We can also display field (column) `@id`s for each record set. The structure of Croissant datasets organizes data as record sets, each of which may correspond to a table or a collection of records.

In [ ]:
# List all record sets in the dataset
record_sets = [rs['@id'] for rs in (getattr(metadata, 'recordSet', []) or [])]
if not record_sets:
    # Try to infer record sets if empty (sometimes a dataset has a single main record set at top level)
    print("No explicit record sets listed in metadata. Attempting to enumerate via dataset.records().")
    # See what recordSet ids are available through the Croissant dataset instance
    try:
        record_set_ids = set()
        for rset in getattr(dataset, '_record_sets', []):
            record_set_ids.add(rset['@id'])
        record_sets = list(record_set_ids)
    except Exception as e:
        print('Could not enumerate record sets:', e)

if not record_sets:
    # Try loading a single main record set (commonly 'main' or similar)
    print('No record sets found.')
else:
    print(f"Found {len(record_sets)} record set(s):")
    for ridx, rsid in enumerate(record_sets):
        print(f"  [{ridx}] @id: {rsid}")
        # Try getting field (column) info for this record set
        # Croissant sometimes encodes columns in metadata as well
        try:
            if hasattr(dataset, '_record_sets_by_id') and rsid in dataset._record_sets_by_id:
                record_set_md = dataset._record_sets_by_id[rsid]
                if 'field' in record_set_md:
                    fields = record_set_md['field']
                    if isinstance(fields, dict):
                        fields = [fields]
                    print('     columns:')
                    for f in fields:
                        col_id = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
                        col_label = f.get('name', '') if isinstance(f, dict) else ''
                        print(f'       - {col_id} {"["+col_label+"]" if col_label else ""}')
        except Exception as e:
            print(f'     Could not retrieve columns: {e}')

## 3. Data Extraction
We will extract data from each record set by their `@id`, and load each into a pandas DataFrame. Refer to the overview above for the `@id` of the main record set containing the mass spectrometry data. We will demonstrate on the first detected record set.

In [ ]:
# Pick the main record set for further analysis
if len(record_sets) == 0:
    raise ValueError("No record sets detected in the dataset.")
# For demonstration, select the first record set
main_record_set_id = record_sets[0]
print(f"Using record set: {main_record_set_id}")

dataframes = {}
for rid in record_sets:
    print(f"Loading records for record set: {rid}")
    records = list(dataset.records(record_set=rid))
    df = pd.DataFrame(records)
    print(f"  Number of records: {len(df)}; Columns: {df.columns.tolist()}")
    dataframes[rid] = df

# Display available columns in the chosen main record set
if main_record_set_id in dataframes:
    print(f"\nColumns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set and perform exploratory operations such as filtering, normalizing a numeric field, and grouping by a key attribute.

Use `@id` strings as DataFrame column names (as per the Croissant schema) for all field references.

In [ ]:
# To proceed, determine a numeric field and a grouping field by inspecting columns
main_df = dataframes[main_record_set_id]
print('Column sample:', main_df.columns[:10].tolist())

# Example: try to pick a numeric field for demonstration, e.g., 'cr:peptide_count' and group by 'cr:description'
# Replace with actual column @id present in your data, e.g. 'cr:peptide_count' if present
import numpy as np

# Try to auto-detect a numeric field
numeric_candidates = main_df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_candidates) == 0:
    # Try to infer from typical Croissant names
    for col in main_df.columns:
        if 'count' in col.lower() or 'abundance' in col.lower() or 'coverage' in col.lower():
            try:
                main_df[col] = pd.to_numeric(main_df[col], errors='coerce')
            except Exception:
                pass
    numeric_candidates = main_df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Numeric field candidates: {numeric_candidates}")

# Select the first numeric field for demonstration
numeric_field = numeric_candidates[0] if numeric_candidates else None
if not numeric_field:
    raise ValueError('No numeric field found for EDA.')

threshold = main_df[numeric_field].mean()  # Filter on mean as example
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the values
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a plausible field (e.g. description or accession)
group_candidates = [c for c in main_df.columns if ('desc' in c.lower() or 'accession' in c.lower() or 'sample' in c.lower())]
group_field = group_candidates[0] if group_candidates else main_df.columns[0]  # default to first column
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (showing means):")
    display(grouped_df.head())

## 5. Visualization
Let's create a simple histogram of the selected numeric field and a boxplot grouped by the group field, where available.

You can modify these plots as appropriate for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the main numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group field if not too many groups
if group_field and main_df[group_field].nunique() <= 20:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant`. We demonstrated how to:

1. List all record sets and their structure using the Croissant schema (`@id` references).
2. Load any record set's data into pandas DataFrames for analysis.
3. Filter and normalize records based on a numeric field (referenced by its `@id`), and group by a descriptive field.
4. Visualize core numeric fields' distributions.

You can continue by choosing other numeric or categorical fields for deeper scientific analysis, leveraging standardized Croissant metadata and Python's rich ecosystem.